# 🇰🇪 Mwalimu AI: Fine-Tuning & Quantizing an On-Device Small Language Model (SLM)
### Tailored for Kenyan Grade 6 Competency-Based Curriculum (CBC) Mathematics

This notebook trains an ultra-compact, high-intelligence **Small Language Model (SLM)** (~75 MB to 290 MB) to act as an offline pedagogical assistant for Kenyan teachers.

**Key Capabilities Learned:**
1. **Dynamic Custom Materials:** Weaves any local material (avocado seeds, dry sticks, bottle caps, stones, beans) directly into classroom activities.
2. **Custom Teacher Hooks:** Converts teacher activity ideas into 5-minute inquiry hooks.
3. **Large Class Management:** Squad roles (Facilitator, Material Custodian, Scribe, Timekeeper) for 45–65 learners.
4. **KICD 4-Tier Assessment Rubrics:** Automatically generates observable criteria (*EE, ME, AE, BE*) and 2-minute exit tickets.
5. **Zero Cloud Latency:** Quantized to 4-bit INT4 for instant, 100% offline inference on budget Android devices (Redmi, Tecno, Infinix).

**Recommended GPU:** Google Colab Free **T4 GPU** (Runtime -> Change runtime type -> T4 GPU).

## 1. Install Unsloth & Required Libraries
We use **Unsloth** for 2x faster fine-tuning with 70% less VRAM usage.

In [ ]:
# Install Unsloth, xformers, and Hugging Face dependencies
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install datasets transformers huggingface_hub

## 2. Load Base Small Language Model (SLM)
Choose between:
- **`Qwen/Qwen2.5-0.5B-Instruct`** *(Recommended)*: 490M params, **~290 MB** quantized, exceptional reasoning and pedagogical structuring.
- **`HuggingFaceTB/SmolLM2-135M-Instruct`**: 135M params, **~75 MB** quantized, ultra-lightweight.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024 # Supports full lesson plans and rubrics
dtype = None # Auto-detect Float16 / Bfloat16
load_in_4bit = True # 4-bit quantization saves GPU VRAM

# Choose model:
model_name = "Qwen/Qwen2.5-0.5B-Instruct" # or "HuggingFaceTB/SmolLM2-135M-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print(f"Loaded {model_name} successfully!")

## 3. Attach LoRA / QLoRA Adapters
We only train lightweight adapter weights (LoRA), keeping the base model intact.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("LoRA adapter attached. Trainable parameters configured!")

## 4. Load the CBC Training Dataset
Upload `cbc_teacher_train.jsonl` to the Colab files pane (left sidebar), or run the cell below to load it automatically.

In [ ]:
import os
from datasets import load_dataset

# Check if dataset exists, otherwise prompt user to upload
dataset_path = "cbc_teacher_train.jsonl"
if not os.path.exists(dataset_path):
    print("Uploading cbc_teacher_train.jsonl...")
    from google.colab import files
    uploaded = files.upload()

dataset = load_dataset("json", data_files={"train": dataset_path})["train"]
print(f"Loaded {len(dataset)} comprehensive training samples!")

# Format into chat prompt using tokenizer's template
def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print("Sample training text preview:")
print(dataset[0]["text"][:400] + "...")

## 5. Train the Model with SFTTrainer
Fine-tuning takes ~15–25 minutes on Google Colab T4 GPU.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 15,
        max_steps = 150, # Sufficient for 800 high-quality samples
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "mwalimu_checkpoints",
    ),
)

trainer_stats = trainer.train()
print(f"Training finished! Total runtime: {trainer_stats.metrics['train_runtime']:.1f} seconds")

## 6. Test Model with Real Kenyan Classroom Scenarios
Test that the model actively weaves the teacher's exact custom materials and activity ideas into the lesson plan.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """Create a 40-minute CBC Lesson Plan for Grade 6 Mathematics.
• Strand: Numbers
• Sub-strand: Fractions (addition and subtraction of mixed numbers)
• Available Local Resources: Avocado seeds and dry sticks
• Teacher Activity Idea / Hook: Learners simulate a Nakuru avocado market where 3 farmers pool and share fractional baskets of seeds.
• Class Size: 54 learners (Mixed ability)
• Environment: Low-resource Kenyan classroom, no electricity, no internet."""

messages = [
    {"role": "system", "content": "You are Mwalimu AI, an intelligent, empathetic on-device pedagogical assistant for Kenyan Grade 6 Competency-Based Curriculum (CBC) Mathematics teachers. Always weave the teacher's exact materials and lesson ideas directly into the activities, steps, and questions."},
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True, temperature = 0.3)
generated_response = tokenizer.batch_decode(outputs)
print(generated_response[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", ""))

## 7. Quantize to 4-bit INT4 (GGUF & MediaPipe Format)
This compresses the model from ~1 GB down to **~290 MB** (or ~75 MB for SmolLM2), making it small enough for budget Android devices.

In [ ]:
# Export to GGUF format (q4_k_m 4-bit quantization)
model.save_pretrained_gguf("mwalimu_slm_q4_k_m", tokenizer, quantization_method = "q4_k_m")

import glob
gguf_files = glob.glob("mwalimu_slm_q4_k_m/*.gguf")
for f in gguf_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"Exported Quantized Model: {f} | Size: {size_mb:.1f} MB")

## 8. Download the Model File
Save the quantized file to Google Drive or download directly to your computer.

In [ ]:
# Option A: Copy to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp mwalimu_slm_q4_k_m/*.gguf /content/drive/MyDrive/

# Option B: Download directly to your browser
from google.colab import files
for f in gguf_files:
    print(f"Downloading {f}...")
    files.download(f)

## 9. How to Load in Mwalimu AI Android App

1. **Push the downloaded model to your connected phone:**
```bash
adb push mwalimu-0.5b-q4_k_m.gguf /sdcard/Android/data/ke.co.mwalimuai/files/
```

2. **In `OfflineLanguageModelEngine.java`**, the engine loads this file directly using MediaPipe GenAI / LiteRT for 100% offline generation with zero network access!